# SMS spam: data prep, classical ML vs a small TensorFlow model
Run from `nlp_serving/` after `./download_data.sh` and `python -m python.run`. Everything here reads artefacts produced by that pipeline, so the numbers match the README, Go parity tests and the R check.

In [1]:
import json, sys, os
sys.path.insert(0, '..'); os.chdir('..')
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from python import data, tokenizer
df = data.load()
df['n_tokens'] = df.text.map(lambda t: len(tokenizer.tokens(t)))
print(df.shape); print(df.label.value_counts())

(5574, 4)
label
ham     4827
spam     747
Name: count, dtype: int64


## Data preparation
Class balance is about 87/13, so accuracy is misleading. We use stratified splitting, `class_weight='balanced'`, and report precision, recall, F1 and AUC.

In [2]:
df.groupby('label').n_tokens.describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4827.0,13.043712,10.449529,0.0,6.0,10.0,17.0,176.0
spam,747.0,23.412316,5.291377,1.0,21.0,25.0,27.0,39.0


In [3]:
tr, te = data.split(df)
print(len(tr), len(te), tr.y.mean().round(4), te.y.mean().round(4))

4459 1115 0.1341 0.1336


## Tokenizer (shared with the Go service)
ASCII-only on purpose, so Python and Go agree byte for byte.

In [4]:
tokenizer.analyze('WINNER!! Claim your FREE prize, call 09061701461 now')

['winner',
 'claim',
 'your',
 'free',
 'prize',
 'call',
 '09061701461',
 'now',
 'winner claim',
 'claim your',
 'your free',
 'free prize',
 'prize call',
 'call 09061701461',
 '09061701461 now']

## Results

In [5]:
m = json.load(open('results/metrics.json'))
pd.DataFrame({k: v for k, v in m.items() if isinstance(v, dict)}).T[['tp','fp','fn','tn','precision','recall','f1','auc']].round(4)

,tp,fp,fn,tn,precision,recall,f1,auc
dataset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tfidf_logreg,140.0,3.0,9.0,963.0,0.9790,0.9396,0.9589,0.9913
tensorflow_keras,146.0,88.0,3.0,878.0,0.6239,0.9799,0.7624,0.9858


The Keras model ranks messages almost as well (AUC) but at threshold 0.5 over-flags ham, because the class weights push its probabilities up. The threshold is a deployment choice, so the next cells look at it.

In [6]:
p = pd.read_csv('results/test_predictions.csv')
from python.metrics import summarise
rows = []
for name, col in [('tfidf_logreg','p_tfidf'),('tensorflow_keras','p_tf')]:
    for thr in (0.3,0.5,0.7,0.9):
        s = summarise(p.y.to_numpy(), p[col].to_numpy(), thr)
        rows.append({'model': name, 'threshold': thr, **{k: round(s[k],3) for k in ('precision','recall','f1')}})
pd.DataFrame(rows)

,model,threshold,precision,recall,f1
0,tfidf_logreg,0.3,0.911,0.960,0.935
1,tfidf_logreg,0.5,0.979,0.940,0.959
2,tfidf_logreg,0.7,0.978,0.913,0.944
3,tfidf_logreg,0.9,1.000,0.852,0.920
4,tensorflow_keras,0.3,0.603,0.980,0.747
5,tensorflow_keras,0.5,0.624,0.980,0.762
6,tensorflow_keras,0.7,0.643,0.980,0.777
7,tensorflow_keras,0.9,0.689,0.966,0.804


In [7]:
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(4.5,4))
for name, col in [('TF-IDF + LogReg','p_tfidf'),('Keras','p_tf')]:
    f, t, _ = roc_curve(p.y, p[col]); ax.plot(f, t, label=name)
ax.plot([0,1],[0,1],'k:'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(); ax.set_title('ROC on held-out 20%')
plt.show()

## Error analysis (baseline false negatives)

In [8]:
te = te.reset_index(drop=True)
fn = te[(te.y==1) & (p.p_tfidf.values < 0.5)]
for t in fn.text.head(5): print('-', t[:140])

- Hi ya babe x u 4goten bout me?' scammers getting smart..Though this is a regular vodafone no, if you respond you get further prem rate msg/s
- ROMCAPspam Everyone around should be responding well to your presence since you are so warm and outgoing. You are bringing in a real breath 
- Sorry I missed your call let's talk when you have the time. I'm on 07090201529
- Can U get 2 phone NOW? I wanna chat 2 set up meet Call me NOW on 09096102316 U can cum here 2moro Luv JANE xx Calls£1/minmoremobsEMSPOBox45P
- Do you realize that in about 40 years, we'll have thousands of old ladies running around with tattoos?
